In [2]:

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"


!pip install --no-deps xformers trl peft transformers accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-fhwprzyw/unsloth_fadf8744aa664c2197c3f6dcca202578
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-fhwprzyw/unsloth_fadf8744aa664c2197c3f6dcca202578
  Resolved https://github.com/unslothai/unsloth.git to commit 27b6d553fe9dcd5a55d6e9e005825c7842e7c014
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 131.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 119.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 23.1 MB/s eta 0:00:00
  

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Modelin hafıza uzunluğu
dtype = None # Donanıma göre otomatik seçim (Float16 / Bfloat16)
load_in_4bit = True # Bellek tasarrufu için 4-bit kuantizasyon aktif

# 1. Taban Modeli Yükleme
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit", # Eğiteceğimiz taban model
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 2. Modeli LoRA/QLoRA Eğitimine Hazırlama
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # LoRA Rank parametresi
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Bellek kullanımını minimize eder
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print(" Taban model ve LoRA katmanları eğitim için hazırlandı!")

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1432: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.7.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


 Taban model ve LoRA katmanları eğitim için hazırlandı!


In [ ]:
from datasets import load_dataset

# 1. Hugging Face'ten animasyon veri setini çekiyoruz
dataset = load_dataset("meldakahramann/animasyon-domain-dataset", split="train")

# 2. Modelin anlayacağı <|im_start|> ve <|im_end|> sohbet şablonu fonksiyonu
def formatting_prompts_func(examples):
    convs = examples["messages"]
    texts = []

    for conv in convs:
        text = ""
        for msg in conv:
            role = msg["role"]
            content = msg["content"]
            thinking = msg.get("thinking", None)

            text += f"<|im_start|>{role}\n"
            # Asistanın iç sesi (thinking) varsa şablona ekliyoruz
            if role == "assistant" and thinking:
                text += f"<|im_start|>thinking\n{thinking}<|im_end|>\n"
            text += f"{content}<|im_end|>\n"

        texts.append(text)

    return { "text" : texts }

# Veri setini dönüştürüyoruz
dataset = dataset.map(formatting_prompts_func, batched=True)
print(" Veri seti chat şablonuna göre başarıyla dönüştürüldü!")

 Veri seti chat şablonuna göre başarıyla dönüştürüldü!


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 1,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # 1.020 satırlık veri setin için en ideal adım sayısıdır
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("Eğitim başlatılıyor...")
trainer_stats = trainer.train()
print("[MÜKEMMEL] Model başarıyla eğitildi!")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1020 [00:00<?, ? examples/s]

Eğitim başlatılıyor...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,020 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,3.328209
2,3.364908
3,3.022553
4,2.941820
5,2.765410
6,2.767314
7,2.532340
8,2.476731
9,2.144501
10,2.201167


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


PicklingError: Can't pickle <class 'trl.trainer.sft_config.SFTConfig'>: it's not the same object as trl.trainer.sft_config.SFTConfig

## Eski Modele göre verilen yanıt
---
filminin seslendirme kadrosunda yer alan başlıca sanatçılar şunlardır: charlie mason, tom hanks, derya kurt, ozden turgut, ayça binici. filminin seslendirme yapımı, dublaj ve mizah kurgusu ise 20th century fox film production şirketi gerçekleştirilmiştir.

filminin ana hikaye yapısı ise şu şekildedir: buz devriyle birlikte dünyayı kaplayan devasa buz tabakalarının erimeye başladığı, tundra'nın arka planda devasa bir devasa buzulun erimeye yüz tuttuğu, buzun eriyip okyanusa döndüğü ve dünyanın iklimi, bitki örtüsü ve canlı yaşamı tamamen değişen, devasa buzulların arkasında kalan çöl vahşi bir tabiatın kapısında duran, koca bir buz adam olan manny (charlie mason) ile eşi elena'nın (derya kurt) bir oğlu olan 3 yaşındaki küçük manny'nin (ozden turgut) büyüsünü ve çocukluğunu korumak için çölde kovalanan, buz devrimi nedeniyle yok olmaya mahkum olan devasa buzulanın son gerileyişini izleyen, kurtlarin (ayça binici) lideri soto'nun (tom hanks) liderlik ettiği, buzullardan geriye kalan son gerilla siperanin sonuna kadar savunan, buzun eriyip okyanusa döndüğü o vahşi, devasa buzun arkasında kalan çöl tabiatının kapısında duran, buz adam manny'nin (charlie mason) büyüsünü ve çocukluğunu korumak için kovalanan gerileyen devasa buzulanin son gerileyişini izleyen, kurtlarin lideri soto'nun (tom hanks) liderlik ettiği buz gerilla siperaninin sonuna kadar savunan, buzun eriyip okyanusa döndüğü o vahşi çöl tabiatinin son gerileyişini izleyen, buzun eriyip okyanusa döndüğü o vahşi çöl tabiatinin son gerileyişini izleyen buz devrimi filminin ana hikaye yapısıdır. filminin senaryo kurgusu ise don bluth, christopher

## İlk Model Çıktısının Analizi ve Yetersizlikler

Yapılan ilk testte modelin çıktısı incelendiğinde üç temel teknik yetersizlik göze çarpmaktadır:

1. **Sonsuz Tekrar Döngüsü:** Model, metnin sonlarına doğru *"buzun eriyip okyanusa döndüğü o vahşi çöl tabiatının son gerileyişini izleyen"* ifadesini art arda tekrarlayarak kilitlenmiştir. Bu durum, modelin durma kriterini (EOS token) belirleyememesinden ve üretilen olasılık dağılımının kısıtlanmasından kaynaklanmaktadır.
2. **Kritik Halüsinasyon:** Model, "Buz Devri" filmiyle tamamen alakasız bilgiler üretmiştir. Manny karakterine eş olarak "Elena", 3 yaşında bir "küçük Manny", Tom Hanks seslendirmesi ve Antarktika/çöl senaryosu gibi gerçek dışı ögeler kurgulamıştır.
3. **Format Uyumsuzluğu:** Eğitim veri setinde modele kazandırılmak istenen `<|im_start|>thinking` iç ses (akıl yürütme) yapısı tamamen göz ardı edilmiş, model doğrudan hatalı bir yanıt üretmeye geçmiştir.

In [ ]:
from transformers import TextStreamer

# Modeli test (inference) moduna alıyoruz
FastLanguageModel.for_inference(model)

# Modelin chat şablonuna uygun bir test sorusu hazırlıyoruz
messages = [
    {"role": "user", "content": "Buz Devri filminin konusu nedir ve seslendirme kadrosunda kimler var?"}
]

# Soruyu tokenizer ile sayılara dönüştürüyoruz
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

# Yanıtın ekrana canlı olarak akması için streamer tanımlıyoruz
text_streamer = TextStreamer(tokenizer, skip_prompt = True)

# Modeli çalıştırıp çıktıyı görüyoruz
print("Modelin Yanıtı:\n")
_ = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 512,
    temperature = 0.5,
    stop_strings = ["<|im_end|>"],
    tokenizer = tokenizer
)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Modelin Yanıtı:



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

filminin seslendirme kadrosunda yer alan başlıca sanatçılar şunlardır: charlie mason, tom hanks, derya kurt, ozden turgut, ayça binici. filminin seslendirme yapımı, dublaj ve mizah kurgusu ise 20th century fox film production şirketi gerçekleştirilmiştir.

filminin ana hikaye yapısı ise şu şekildedir: buz devriyle birlikte dünyayı kaplayan devasa buz tabakalarının erimeye başladığı, tundra'nın arka planda devasa bir devasa buzulun erimeye yüz tuttuğu, buzun eriyip okyanusa döndüğü ve dünyanın iklimi, bitki örtüsü ve canlı yaşamı tamamen değişen, devasa buzulların arkasında kalan çöl vahşi bir tabiatın kapısında duran, koca bir buz adam olan manny (charlie mason) ile eşi elena'nın (derya kurt) bir oğlu olan 3 yaşındaki küçük manny'nin (ozden turgut) büyüsünü ve çocukluğunu korumak için çölde kovalanan, buz devrimi nedeniyle yok olmaya mahkum olan devasa buzulanın son gerileyişini izleyen, kurtlarin (ayça binici) lideri soto'nun (tom hanks) liderlik ettiği, buzullardan geriye kalan son g

## İkinci Eğitimde Yapılan Değişiklikler ve Model Çıktısının Değerlendirilmesi

İlk modeldeki kilitlenme ve kaydetme süreçlerindeki hataları (`PicklingError` ve `multiprocessing` deadlock) aşmak için eğitim mimarisinde şu güncellemeler yapılmıştır:
* **Veri İşleme (Process) Ayarı:** İşlemci kilitlenmelerini önlemek adına `dataset_num_proc = 1` olarak sabitlenmiştir.
* **Kaydetme Stratejisi:** Otomatik kontrol noktası (checkpoint) kaydetme çakışmalarını engellemek için `save_strategy = "no"` parametresi eklenmiştir.
* **Format Hizalaması:** Veri setindeki sohbet şablonu  işleme fonksiyonu gözden geçirilmiştir.



In [ ]:
import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# 1. Ayarlar
max_seq_length = 2048
dtype = None
load_in_4bit = True

# 2. Model ve Tokenizer Yükleme
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 3. LoRA Ayarları
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

# 4. Veri Setini Yükleme ve Chat Şablonuna Dönüştürme
dataset = load_dataset("meldakahramann/animasyon-domain-dataset", split="train")

def formatting_prompts_func(examples):
    convs = examples["messages"]
    texts = []
    for conv in convs:
        text = ""
        for msg in conv:
            role = msg["role"]
            content = msg["content"]
            thinking = msg.get("thinking", None)

            text += f"<|im_start|>{role}\n"
            if role == "assistant" and thinking:
                text += f"<|im_start|>thinking\n{thinking}<|im_end|>\n"
            text += f"{content}<|im_end|>\n"
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched=True)

# 5. Eğitim Döngüsü
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 1, # Kilitlenmeyi önler
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "no", # PicklingError hatasını tamamen engeller
    ),
)

print("Eğitim yeniden başlatılıyor...")
trainer_stats = trainer.train()
print(" Model  eksiksiz ve hatasız eğitildi!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.7.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Map:   0%|          | 0/1020 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1020 [00:00<?, ? examples/s]

Eğitim yeniden başlatılıyor...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,020 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,3.328209
2,3.364908
3,3.022834
4,2.941829
5,2.765314
6,2.767256
7,2.532640
8,2.476745
9,2.144531
10,2.201177


 Model  eksiksiz ve hatasız eğitildi!


## Tekrar Bastan Eğitilen modele göre çıktı
---
filminin seslendirme kadrosunda yer alan başlıca sanatçılar şunlardır: daveed diggs, neve campbell, ronan quinn, josh charbo, oded fehr, kadir dar. filminin seslendirme sanatçılarından bahsetmeden önce hikayesini net bir dille ve mahremiyete riayet ederek sunmalıyım.<|im_start|>asker
milyonlarca yıl önce, buz devriyle birlikte yeryüzü buzlarla kaplı bir cehennem gibi bir durumdadır. yeryüzünün son insan kolonisi olan antarktika'da, bilim insanları dr. neander ve dr. soto, insan ırkı için bir son çare bulmak üzere devasa bir buz kırıcı inşa etmişlerdir. buz kırıcının ilk denemesini gerçekleştiren ekipteki iki yetişkin çocuk, siberiya'da buzun altında saklanan, insan ırkının evrim geçirmesi için son şans olan son insan kabilelerini bulmak üzere buzun derinliklerine doğru yolculuğa çıkarlar. bu yolculuk esnasında, buzun derinliklerinde saklanan devasa ve korkunç canavarlarla yüzleşen çocuklar, buzun altındaki gizemli bir dünya keşfedeceklerdir.<|im_end|>

##  İkinci Model Çıktısının Analizi ve Kalan Eksiklikler

* **Gelişen Yönler:** Model, ilk denemedeki sonsuz metin tekrarı  sorununu aşmış ve akıcı bir cümle yapısı yakalamıştır. En önemlisi, veri setindeki düşünme/şablon yapısını hatırlayarak `<|im_start|>` ve `<|im_end|>` etiketlerini kullanmaya başlamış, yanıt vermeden önce bir iç ses ("hikayesini net bir dille sunmalıyım") kurgulamıştır.
* **Kalan Eksiklikler:** Model hâlâ ciddi düzeyde halüsinasyon üretmektedir (Buz kırıcı gemiler, Antarktika insan kolonisi vb.). Ayrıca şablon etiketleri model tarafından henüz "özel komut" olarak tam anlamıyla özümsenmediği için `<|im_start|>asker` gibi hatalı rol tanımlamaları ortaya çıkmıştır.

In [ ]:
from transformers import TextStreamer

# Modeli test (inference) moduna alıyoruz
FastLanguageModel.for_inference(model)

# Modelin chat şablonuna uygun bir test sorusu hazırlıyoruz
messages = [
    {"role": "user", "content": "Buz Devri filminin konusu nedir ve seslendirme kadrosunda kimler var?"}
]

# Soruyu tokenizer ile sayılara dönüştürüyoruz
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

# Yanıtın ekrana canlı olarak akması için streamer tanımlıyoruz
text_streamer = TextStreamer(tokenizer, skip_prompt = True)

# Modeli çalıştırıp çıktıyı görüyoruz
print("Modelin Yanıtı:\n")
_ = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 512,
    temperature = 0.5,
    stop_strings = ["<|im_end|>"],
    tokenizer = tokenizer
)

Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Modelin Yanıtı:

filminin seslendirme kadrosunda yer alan başlıca sanatçılar şunlardır: daveed diggs, neve campbell, ronan quinn, josh charbo, oded fehr, kadir dar. filminin seslendirme sanatçılarından bahsetmeden önce hikayesini net bir dille ve mahremiyete riayet ederek sunmalıyım.<|im_start|>asker
milyonlarca yıl önce, buz devriyle birlikte yeryüzü buzlarla kaplı bir cehennem gibi bir durumdadır. yeryüzünün son insan kolonisi olan antarktika'da, bilim insanları dr. neander ve dr. soto, insan ırkı için bir son çare bulmak üzere devasa bir buz kırıcı inşa etmişlerdir. buz kırıcının ilk denemesini gerçekleştiren ekipteki iki yetişkin çocuk, siberiya'da buzun altında saklanan, insan ırkının evrim geçirmesi için son şans olan son insan kabilelerini bulmak üzere buzun derinliklerine doğru yolculuğa çıkarlar. bu yolculuk esnasında, buzun derinliklerinde saklanan devasa ve korkunç canavarlarla yüzleşen çocuklar, buzun altındaki gizemli bir dünya keşfedeceklerdir.<|im_end|>


## Ratatuy Filmi İçin Denedik Tekrar
---
Modelin Yanıtı:

Ratatuy filminin olay örgüsü şu şekildedir: 18. yüzyılın sonbaharında, Paris'in ünlü gurme restoranlarından olan Gusteau'nun (Pomme de Terre) şefi Auguste Gusteau, en büyük hayali olan 'Kazançlı Yemeğin Yüce Mottosu' olan 'Yemeğin Bedeli Yoktur' (Anyone Can Cook) sloganını geride bırakmış ve devasa bir restoran ağı kurmuştur. Gusteau'nun en yakın dostları ve en iyi aşçılarından olan, genç ve yetenekli Antonio ve Colette, şefin ölümüyle birlikte restoranın başına geçerler. Ancak restoranın gerçek sahibi olan kurnaz ve acımasız patron, Gusteau'nun ölümünün ardından restorana gizli bir koruma sistemi kurar. Bu koruma sisteminin adı 'Ratatuy' (Remy) ve amacı, restorana sızan sıçanların (mice) restoranda çalışanlar tarafından fark edilmesini ve kovulmasını sağlamaktır.

Filmin ana kahramanı olan Remy, bir sıçan olan ancak aşçılık sanatını son derece iyi yapan ve Gusteau'nun şeflik kitabını ezberleyerek, restorana gizlice girmiş ve şeflik makamına hakim olmuştur. Remy, restoranda çalışanlar tarafından fark edilmeden yemek hazırlamak için, şeflik odasında çalışan ve kör olan şef Linguini ile işbirliği yapar. Linguini, Remy'nin sesini duymakta ve onun emrettiği gibi yemek hazırlamaktadır. Ancak restoranda çalışanlar, Remy'nin şeflik odasında gizlendiğini fark eder ve onu kovmak için bir plan yaparlar. Remy, restoranda çalışanın kızı Colette ile dost olur ve onun şeflik sanatına dair sırları öğrenir. Remy, restorana sızan sıçanların (mice) restoranda çalışanlar tarafından fark edilmesini engellemek için, Colette ile birlikte restoranda gizli bir aşçılık ekibi kurar. Bu ekibe katılan diğer sıçanlar ise, Remy'nin babası Django, amcası Emile, amcasının kızı ve Remy'nin kız kar

## Ratatuy Denemesinin Analizi

Modelin veri setinde doğrudan yer alan bir film üzerinden test edilmesi sonucunda elde edilen bulgular:

1. **Domain (Alan) Bilgisinin Yakalanması:** Model bu kez tamamen uydurma bir evren yaratmak yerine, eğitildiği veri setindeki alan bilgisini  kullanmıştır. *Gusteau*, *Anyone Can Cook*, *Remy*, *Linguini*, *Colette*, *Django* ve *Emile* gibi gerçek film karakterlerini ve terimlerini doğru şekilde çağırmıştır.
2. **Karakter ve Olay Örgüsü Karmaşası:** Veri setindeki terimler yakalanmış olsa da olay örgüsü birbirine karışmıştır. Linguini "kör bir şef" olarak tanımlanmış, *Ratatuy* ismi sıçan kovma sistemi olarak kurgulanmış ve Gusteau'nun adına Fransızca "patates" anlamına gelen *(Pomme de Terre)* eklenmiştir.
3. **İç Ses (Thinking) Etiketinin Atlanması:** Model, bu yanıtında düşünme aşamasını tamamen atlayarak doğrudan cevaba geçmiştir.

**Sonuç:** Model, veri setindeki kelimeleri ve kavramları öğrenmiş (Domain Adaptation gerçekleşmiş) ancak şablon biçimini ve olay akışını tam oturtamamıştır.

In [ ]:
from transformers import TextStreamer

# Modeli test (inference) moduna alıyoruz
FastLanguageModel.for_inference(model)

# Modelin chat şablonuna uygun bir test sorusu hazırlıyoruz
messages = [
    {"role": "user", "content": "Ratatuy animasyonu neyi anlatıyor, hikayesi nasıl?"}
]

# Soruyu tokenizer ile sayılara dönüştürüyoruz
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

# Yanıtın ekrana canlı olarak akması için streamer tanımlıyoruz
text_streamer = TextStreamer(tokenizer, skip_prompt = True)

# Modeli çalıştırıp çıktıyı görüyoruz
print("Modelin Yanıtı:\n")
_ = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 512,
    temperature = 0.5,
    stop_strings = ["<|im_end|>"],
    tokenizer = tokenizer
)

Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Modelin Yanıtı:

Ratatuy filminin olay örgüsü şu şekildedir: 18. yüzyılın sonbaharında, Paris'in ünlü gurme restoranlarından olan Gusteau'nun (Pomme de Terre) şefi Auguste Gusteau, en büyük hayali olan 'Kazançlı Yemeğin Yüce Mottosu' olan 'Yemeğin Bedeli Yoktur' (Anyone Can Cook) sloganını geride bırakmış ve devasa bir restoran ağı kurmuştur. Gusteau'nun en yakın dostları ve en iyi aşçılarından olan, genç ve yetenekli Antonio ve Colette, şefin ölümüyle birlikte restoranın başına geçerler. Ancak restoranın gerçek sahibi olan kurnaz ve acımasız patron, Gusteau'nun ölümünün ardından restorana gizli bir koruma sistemi kurar. Bu koruma sisteminin adı 'Ratatuy' (Remy) ve amacı, restorana sızan sıçanların (mice) restoranda çalışanlar tarafından fark edilmesini ve kovulmasını sağlamaktır.

Filmin ana kahramanı olan Remy, bir sıçan olan ancak aşçılık sanatını son derece iyi yapan ve Gusteau'nun şeflik kitabını ezberleyerek, restorana gizlice girmiş ve şeflik makamına hakim olmuştur. Remy, resto

## Nihai Model Eğitimi: Özel Token Tanımlaması ve Kapsamlı Eğitim

Önceki denemelerdeki halüsinasyon, şablon atlama ve bilgi karmaşası sorunlarını kökten çözmek amacıyla iki temel teknik iyileştirme birleştirilerek nihai eğitim başlatılmıştır:

1. **Özel Token Entegrasyonu (`add_special_tokens`):**
   * `<|im_start|>`, `<|im_end|>` ve `thinking` ifadeleri modele özel komut karakterleri (Special Tokens) olarak tanıtılmış ve modelin kelime gömme katmanı (`resize_token_embeddings`) yeniden boyutlandırılmıştır. Böylece model bu ifadeleri düz metin olarak değil, sistem talimatı ve düşünme aşaması olarak algılayacaktır.

2. **Adım Sayısının Artırılması (`max_steps = 250`):**
   * 60 adımlık eğitim, modelin veri setindeki bilgileri derinlemesine kavraması için yetersiz kalmıştır. Eğitim adımı **250 adıma** (yaklaşık 2.5 epoch / tam tur) çıkarılarak modelin veri setini yüzeysel öğrenmek yerine hem sohbet formatını hem de doğru film bilgilerini eksiksiz ezberlemesi hedeflenmiştir.

In [3]:
import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# 1. Ayarlar
max_seq_length = 2048
dtype = None
load_in_4bit = True

# 2. Model ve Tokenizer Yükleme
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# KRİTİK DÜZELTME Chat şablonu etiketlerini özel token olarak sisteme tanıtıyoruz
# Böylece model bunları düz metin sanmayacak, özel komut olarak algılayacak.
special_tokens = ["<|im_start|>", "<|im_end|>", "thinking"]
tokenizer.add_special_tokens({"additional_special_tokens": special_tokens})
model.resize_token_embeddings(len(tokenizer))

# 3. LoRA Ayarları
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

# 4. Veri Setini Yükleme ve Chat Şablonuna Dönüştürme
dataset = load_dataset("meldakahramann/animasyon-domain-dataset", split="train")

def formatting_prompts_func(examples):
    convs = examples["messages"]
    texts = []
    for conv in convs:
        text = ""
        for msg in conv:
            role = msg["role"]
            content = msg["content"]
            thinking = msg.get("thinking", None)

            text += f"<|im_start|>{role}\n"
            if role == "assistant" and thinking:
                text += f"<|im_start|>thinking\n{thinking}<|im_end|>\n"
            text += f"{content}<|im_end|>\n"
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched=True)

# 5. Eğitim Döngüsü
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 1,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 250, # Modelin bilgileri ve şablonu tam oturtması için 250 adıma çıkardık
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5, # Ekranın loglarla dolup taşmaması için 5 adımda bir yazdırıyoruz
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "no", # Çakışma hatasını engeller
    ),
)

print("Kapsamlı eğitim başlatılıyor")
trainer_stats = trainer.train()
print(" Model başarıyla eğitildi!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
Unsloth 2026.7.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


README.md:   0%|          | 0.00/467 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  145kB            

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,020 | Num Epochs = 2 | Total steps = 250
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,220,672 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,3.742780
10,3.221191
15,2.730660
20,2.495016
25,2.154125
30,2.008522
35,1.871320
40,1.908270
45,1.637504
50,1.741674


Step,Training Loss
5,3.742780
10,3.221191
15,2.730660
20,2.495016
25,2.154125
30,2.008522
35,1.871320
40,1.908270
45,1.637504
50,1.741674


 Model başarıyla eğitildi!


## NİHAİ MODELİN YANITI

<|eot_id|>
Kullanıcı hem filminin senaryo yapısını hem de seslendirme kadrosunu öğrenmek istedi. Özet hikaye ile seslendirme sanatçılarının isimlerini tek bir bütünleşik asistan cevabı haline getirmeliyim.<|eot_id|>
Dünya, tek bir büyük buz kütlesinden ibaret ve tüm canlıların hayatını tecellii bir buz barı içinde sürdürdüğü, son dondurucu kışın ortasında veda etmiş ve sıcak bir yeşillik dönemevinin el ele geçtiği dar bir coğrafa sahiptir. Bu dondurucu soğuklık bölgesinde, şans eseri bir göletinin geçtiği yerin etrafında havlayarak hayatına devam eden çılgın bir hortumlardan (Sid) olan Manny, kendisinin dünyanın son kalan hayvanı olduğunu düşünen evhamlı ve öfkeli yaban geyiği Manny, bir kılıç dişli kaplanın (Soto) liderliğindeki acımasız av partisine kaçınılmaz bir şekilde düşer. Ormanın derinliklerinde yer alan ve koruyucusu olan çılgın tembel mamut Manny, avdan kurtulmayı başarır ancak bu esnada kolonunun yolunu kaybeder ve yalnız bırakılmaktadır. Karanlık ve tehlikeli junglede (çölde) yalnız lazerli bir vahşi boar (Ellie) tarafından avlanmak tehdidi altındaki Manny, yola çıkan şapşal ve eğlenceli tembel mamut Manny'nin peşini geride bırakmadan önce kolonunun büyük bir sakarlık sonucu yüzlerce yıl geride bırakıldığını fark eder. Koloninin yollarına kavuşarak geride bırakılan bu üç dışlanmış canlı (Manny, Sid ve Diego), kendilerini kuruyan kuru çöl vahşi doğasında amansız bir av cennetine (Death Lagoon) sıkışmıştır. Burada hayatta kalabilmek için büyük bir av partisine katılmek zorunda kalan bu üç dışlanmış canlı, avcılara karşı ortak bir takımda yürüyüşe çıkacaktır.

Filmin seslendirme kadrosunda yer alan başlıca sanatçılar ise şunlardır: Ali Poyrazoğlu, Haluk Bilginer, Yekta Kopan, Altan Erkekli, Ray Romano, John Leguizamo, Denis Leary.<|eot_id|>



## Buz Devri Testi ve Alan Bilgisi  Doğrulaması

250 adımlık yeni model ile gerçekleştirilen Buz Devri testinde elde edilen sonuçlar:

1. **Gerçek Veri Tespiti (Dublaj Kadrosu):** Model, önceki denemelerdeki halüsinasyon isimlerin aksine Türkiye'deki gerçek dublaj kadrosunu (*Ali Poyrazoğlu, Haluk Bilginer, Yekta Kopan, Altan Erkekli*) %100 doğrulukla çağırmıştır. Bu durum, LoRA eğitiminin hedef veri setindeki spesifik bilgileri modele kazandırdığını doğrulamaktadır.
2. **Akıl Yürütme:** Düşünme etiketi içerisinde kullanıcının iki aşamalı isteği (senaryo + seslendirme) analiz edilmiş ve yanıt buna göre planlanmıştır.
3. **Senaryo Katmanındaki Karışıklık:** Seslendirme kadrosu tam doğru verilmesine rağmen, senaryo anlatımında karakter isimleri ve türleri (Manny / Sid / Ellie) birbiriyle çakışmıştır. Çıkarım  aşamasındaki `temperature` parametresinin düşürülmesiyle bu kelime kaymalarının önüne geçilebilir.

In [9]:
from transformers import TextStreamer

# Modeli test moduna alıyoruz
FastLanguageModel.for_inference(model)

# Prompt ve Özel Token Tanımları
prompt = "<|im_start|>user\nBuz Devri filminin konusu nedir ve seslendirme kadrosunda kimler var?<|im_end|>\n<|im_start|>assistant\n"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt=True)

# Bitiş token'ını açıkça belirtiyoruz
eos_token_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

print("--- NİHAİ MODELİN YANITI ---\n")
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 512,
    temperature = 0.6,
    top_p = 0.9,
    eos_token_id = eos_token_id, # Modelin duracağı yeri açıkça belirttik
    pad_token_id = tokenizer.pad_token_id,
    tokenizer = tokenizer
)

Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- NİHAİ MODELİN YANITI ---

<|eot_id|>
Kullanıcı hem filminin senaryo yapısını hem de seslendirme kadrosunu öğrenmek istedi. Özet hikaye ile seslendirme sanatçılarının isimlerini tek bir bütünleşik asistan cevabı haline getirmeliyim.<|eot_id|>
Dünya, tek bir büyük buz kütlesinden ibaret ve tüm canlıların hayatını tecellii bir buz barı içinde sürdürdüğü, son dondurucu kışın ortasında veda etmiş ve sıcak bir yeşillik dönemevinin el ele geçtiği dar bir coğrafa sahiptir. Bu dondurucu soğuklık bölgesinde, şans eseri bir göletinin geçtiği yerin etrafında havlayarak hayatına devam eden çılgın bir hortumlardan (Sid) olan Manny, kendisinin dünyanın son kalan hayvanı olduğunu düşünen evhamlı ve öfkeli yaban geyiği Manny, bir kılıç dişli kaplanın (Soto) liderliğindeki acımasız av partisine kaçınılmaz bir şekilde düşer. Ormanın derinliklerinde yer alan ve koruyucusu olan çılgın tembel mamut Manny, avdan kurtulmayı başarır ancak bu esnada kolonunun yolunu kaybeder ve yalnız bırakılmaktadır. Karan

#### Ratatuy Çıktısının Karşılaştırmalı Analizi ve Değerlendirmesi

**Soru:** *Ratatuy animasyonu neyi anlatıyor, hikayesi nasıl?*

---

##### 1.  Eski Çıktılarla Karşılaştırma (Neler Düzeldi?)

* **Halüsinasyonun ve Yanlış Bilgilerin Temizlenmesi:**
  * *Eski Çıktı:* Linguini'yi "kör bir şef" yapıyor, Gusteau'nun adını Fransızca patates anlamına gelen *(Pomme de Terre)* olarak değiştiriyor ve *Ratatuy* kelimesini sıçan kovma mekanizması olarak kurguluyordu.
  * *Yeni Çıktı:* Hikayedeki tüm temel kavramlar yerli yerine oturdu. Remy'nin koku duyusu, Gusteau'nun *"Herkes yemek yapabilir"* mottosu, sakar çöpçü Linguini ile yapılan şapkanın altından saç çekme anlaşması ve gurme eleştirmen Anton Ego %100 doğrulukla ifade edildi.
* **Akıl Yürütme (Reasoning) Entegrasyonu:**
  * Model, yanıtı üretmeden önce `<|eot_id|>` blokları içerisine hedeflediğimiz düşünme adımlarını eklemeyi başardı:
    > *"Ratatuy animasyonunun olay örgüsü istendi. Hafızamdaki özet metne sadık kalarak net ve anlaşılır bir anlatım sunmalıyım..."*

---

#####  2. Kalan Eksiklikler ve Teknik Problemler

Sözdizimi ve alan bilgisi seviyesinde büyük gelişme kaydedilmiş olsa da çıktıda hâlâ çözülmesi gereken bazı teknik problemler yer almaktadır:

1. **Özel Token Hiza Kayması (Token Misalignment):**
   * Model, eğittiğimiz özel etiketler (`<|im_start|>` / `<|im_end|>`) yerine Llama-3'ün varsayılan `<|eot_id|>` ve `<|end_of_text|>` etiketlerini kullanmaya devam etmektedir. Bu durum, modelin stop token ayarını tam yakalayamamasına ve yanıt sonlandırma kriterini doğru uygulayamamasına yol açmaktadır.
2. **Sorulmayan Konulara Kayma :**
   * Hikaye anlatımı kusursuz tamamlanmış olmasına rağmen model durmamış; kendi kendine *"Horton filminin herhangi bir ödülü var mı?"* gibi alakasız veri seti başlıklarını çağırmıştır.
3. **Cümlenin Yarım Kalması :**
   * Kendi kendine *"Ratatuy filminin seslendirenleri kimlerdir?"* sorusunu sorup seslendirme kadrosuna geçmiş, ancak yanıtı *"Evin At..."* şeklinde yarım bırakmıştır. Bu durum `max_new_tokens` limitinden veya özel bitiş token'ının devreye girmemesinden kaynaklanmaktadır.
4. **Çeviri ve İmla Hataları :**
   * Hikaye anlatımı genel olarak akıcı olsa da *"zekası ve yaratıcılığı ise tavanı döşememiş olan fena yemek sevien"* gibi bozuk cümle yapıları ve yabancı kaynaktan doğrudan kelime kelime çevrilememiş ifadeler mevcuttur.

---

#####  Genel Sonuç

250 adımlık yeni eğitim, modelin **veri setindeki doğru hikayeyi ve düşünme (thinking) şablonunu öğrendiğini** açıkça kanıtlamıştır. Karakterlerin ve olayların doğruluğu açısından en başarılı seviyeye ulaşılmıştır. Kalan "kendiliğinden soru sorma" ve "metnin yarım kalması" sorunları, test aşamasındaki `temperature` parametresinin düşürülmesi (`0.2`) ve durdurma token'larının (`stop_strings`) daha sıkı tanımlanmasıyla kontrol altına alınabilir.

In [8]:
from transformers import TextStreamer

# Modeli test moduna alıyoruz
FastLanguageModel.for_inference(model)

# Prompt ve Özel Token Tanımları
prompt = "<|im_start|>user\nRatatuy animasyonu neyi anlatıyor, hikayesi nasıl?<|im_end|>\n<|im_start|>assistant\n"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt=True)

# Bitiş token'ını açıkça belirtiyoruz
eos_token_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

print("--- NİHAİ MODELİN YANITI ---\n")
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 512,
    temperature = 0.6,
    top_p = 0.9,
    eos_token_id = eos_token_id, # Modelin duracağı yeri açıkça belirttik
    pad_token_id = tokenizer.pad_token_id,
    tokenizer = tokenizer
)

Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- NİHAİ MODELİN YANITI ---

<|eot_id|>
Ratatuy animasyonunun olay örgüsü istendi. Hafızamdaki özet metne sadık kalarak net ve anlaşılır bir anlatım sunmalıyım..<|eot_id|>
Remy, koku ve tat alma duyuları olağanüstü derecede gelişmiş, zekası ve yaratıcılığı ise tavanı döşememiş olan fena yemek sevien, dışlanmış bir faredir. Diğer farelerin aksine çöplerle beslenmeyi reddeden ve en büyük idolü olan efsanevi Fransız şef Auguste Gusteau'nun 'Herkes yemek yapabilir' felsefesine inanan Remy'nin hayatı, yaşadığı koloninin dağılmasıyla altüst olur. Şans eseri kendini Paris'in kalbinde, Gusteau'nun ünlü restoranının altında bulan Remy, mutfakta saklanırken işe yeni girmiş olan sakar, özgüvensiz ve yemek yapmaktan zerre anlamayan çöpçü çocuk Linguini'nin harika bir çorbayı mahvettiğini görür. Remy çorbayı gizlice kurtarır ve ortaya çıkan lezzet büyük bir hayranlık uyandırır. Mutfakta farelerin barınması yasak olduğundan, Remy ve Linguini sıra dışı bir gizli işbirliği anlaşması yaparlar: Remy, L

In [10]:
import os
from google.colab import userdata


os.environ["HF_TOKEN"] = userdata.get('huggingface')


repo_id = "meldakahramann/animasyon-lora-adapter"

print("Eğitilen LoRA adaptörü Hugging Face'e yükleniyor, lütfen bekleyin...")

# Modeli ve Tokenizer'ı Hugging Face'e push ediyoruz
model.push_to_hub_merged(
    repo_id,
    tokenizer,
    save_method = "lora",
    token = os.environ["HF_TOKEN"]
)

print(f"\n TEBRİKLER! Modelin başarıyla yayınlandı:")
print(f"https://huggingface.co/{repo_id}")

Eğitilen LoRA adaptörü Hugging Face'e yükleniyor, lütfen bekleyin...


config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in meldakahramann/animasyon-lora-adapter/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0004-of-00004.safetensors:   1%|1         | 15.9MB / 1.17GB            



Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [10:26<00:00, 156.62s/it]


Unsloth: Merge process complete. Saved to `/content/meldakahramann/animasyon-lora-adapter`

 TEBRİKLER! Modelin başarıyla yayınlandı:
https://huggingface.co/meldakahramann/animasyon-lora-adapter
